[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZacharyHillman18/PharmapyWorkshop/blob/main/Workshop_Notebooks_Advanced/CrystallizersOptimization.ipynb)

# Crystallizer optimization with the full crystal size distribution

> **Advanced version.** Fewer lines are filled in than in the standard version, but the notebook has the same structure, names and functions, so you can switch between the two versions at any notebook. The [SyntaxExamples notebook](https://colab.research.google.com/github/ZacharyHillman18/PharmapyWorkshop/blob/main/Workshop_Notebooks_Advanced/SyntaxExamples.ipynb) shows the syntax of every PharmaPy object used here.

In this notebook we use PharmaPy's crystallizer models to find operating conditions that give a good **crystal size distribution (CSD)**: large crystals *and* a narrow distribution.

* **Part A**: optimize a seeded **batch** cooling crystallizer.
* **Part B**: compare the batch product with **MSMPRs** (continuous crystallizers, one or two in series).
* **Part C**: switch on **secondary nucleation** and see which process suffers more.

Compound **C** crystallizes from the solvent. The kinetics are based on PharmaPy's `crystTester.py` (primary nucleation, size-independent growth and an Apelblat solubility).

### Model
The population balance for a well-mixed crystallizer with size-independent growth is

$$ \frac{\partial n}{\partial t} + G \frac{\partial n}{\partial L} = \frac{Q}{V} \left( n_{in} - n \right), \qquad n(L_0, t) = \frac{B}{G} $$

with power-law kinetics on the relative supersaturation $S = (c - c^*)/c^*$:

$$ B = k_b S^b + k_s S^{s_1} (k_v \mu_3)^{s_2}, \qquad G = k_g S^g, \qquad c^* = \exp\left(a_1 + \frac{a_2}{T} + a_3 \ln T\right) $$

($B$ is primary + secondary nucleation and $\mu_3$ the third moment of the CSD.) For a batch unit $Q = 0$. PharmaPy solves the population balance with the **finite-volume method (1D-FVM)**: $n(L)$ is discretized on a size grid (here 60 geometric cells between 1 and 1500 $\mu$m), which gives the full CSD. A batch simulation takes a couple of seconds and an MSMPR start-up to steady state 10-20 s, so we keep the optimization budgets small.

### Product quality metrics
We use the **volume-weighted** distribution: $d_{50}$ is the median size by volume, and the **span** measures its width:

$$ \mathrm{span} = \frac{d_{90} - d_{10}}{d_{50}} $$

**Running on Google Colab?** Run the next cell first. It installs PharmaPy and puts this notebook's data files and `workshop_utils.py` on the Colab machine (about half a minute). Run it again whenever the runtime restarts. To keep your work, use *File → Save a copy in Drive*. On your own computer the cell skips the install and only unpacks the files.

In [ ]:
# @title Colab setup: run this cell first { display-mode: "form" }
# Installs PharmaPy and unpacks workshop_utils.py and data/ into the working
# directory. Run it again after a runtime restart or a factory reset.
import base64, importlib.metadata, importlib.util, io, os, subprocess, sys, zipfile

if 'google.colab' in sys.modules or importlib.util.find_spec('PharmaPy') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pharmapy-org>=0.3.1'], check=True)

_FILES = (
    '...'
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_FILES))) as archive:
    archive.extractall('.')
importlib.invalidate_caches()

try:
    _version = importlib.metadata.version('pharmapy-org')
except importlib.metadata.PackageNotFoundError:   # a development install of PharmaPy
    _version = '(development install)'
print('PharmaPy', _version, '- workshop files ready in', os.getcwd())

## Setup

`workshop_utils.py` (in this folder) contains helpers for the workshop: `csd_quantiles` computes $d_{10}$, $d_{50}$, $d_{90}$ and the span of a CSD, and `volume_density` gives the volume-weighted CSD for plotting. `apply_pharmapy_patches()` checks your PharmaPy version and fixes a few crystallizer bugs of older versions; with a current PharmaPy it does nothing. All units use `ScipyBackend`, which only needs scipy.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from PharmaPy.Phases_Refactored import LiquidPhase, SolidPhase
from PharmaPy.Streams_Refactored import LiquidStream
from PharmaPy.Crystallizers_Refactored import BatchCrystallizer, ContinuousCrystallizer
from PharmaPy.Mechanisms import OneDFVMMechanism
from PharmaPy.Kinetics import CrystKinetics
from PharmaPy.ProcessControl_Refactored import SimpleTemperatureController, ContinuousVesselController
from PharmaPy.IntegratorBackends import ScipyBackend
from PharmaPy.SimExec import SimulationExec

from workshop_utils import apply_pharmapy_patches, csd_quantiles, volume_density

apply_pharmapy_patches()

path = 'data/compounds_workshop.json'   # species: A, B, C, D, solvent

## Crystallization kinetics and solubility

`CrystKinetics` receives the solubility coefficients and the kinetic parameters as tuples:

* `nucl_prim = (k_b, E_b, b)`: primary nucleation, #/(m$^3$ s)
* `nucl_sec = (k_s, E_s, s_1, s_2)`: secondary nucleation, $B_s = k_s S^{s_1} (k_v \mu_3)^{s_2}$ (switched off until Part C)
* `growth = (k_g, E_g, g)`: growth rate, $\mu$m/s
* `solubility_type='apelblat'` with `solubility_basis='mass_per_volume_solvent'`: $c^*$ in kg of C per m$^3$ of solvent

A function returns a **new** kinetics object each time, because every crystallizer needs its own.

In [ ]:
SOLUB_COEFFS = np.array([-28.13909202, 0.001, 5.900800253])   # Apelblat (crystTester)
NO_SECONDARY = (0, 0, 0, 0)


def cryst_kinetics(nucl_sec=NO_SECONDARY):
    return CrystKinetics(SOLUB_COEFFS,
                         nucl_prim=(1.0448e17, 0, 8.30671806),   # #/(m3 s)
                         nucl_sec=nucl_sec,                      # (k_s, E_s, s1, s2)
                         growth=(2.4297e4, 0, 4.52241037),       # um/s
                         solubility_type='apelblat',
                         solubility_basis='mass_per_volume_solvent')


W_C = 0.2               # 20 wt% C in the charge / feed
RHO_SOLVENT = 887.6     # kg/m3 (rho_liq of the solvent in the json file)
X_GRID = np.geomspace(1, 1500, 60)   # um

conc_feed = W_C / (1 - W_C) * RHO_SOLVENT   # kg C / m3 solvent

temps = np.linspace(268, 318, 50)
solub = [cryst_kinetics().get_solubility(T) for T in temps]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(temps, solub, label='solubility $c^*$')
ax.axhline(conc_feed, color='k', ls='--', label='20 wt% charge')
ax.set_xlabel('T (K)')
ax.set_ylabel('kg C / m$^3$ solvent')
ax.legend()
plt.show()

The 20 wt% solution is saturated at about 294 K. Cooling it to 273 K (where $c^* \approx 143$ kg/m$^3$) crystallizes about one third of C. **The final temperature fixes the yield**, so in this notebook we only change *how* we get there.

---
# Part A: Seeded batch cooling crystallizer

**Operation**: 1 kg of 20 wt% C solution is seeded at $T_0 = 293.15$ K (slightly supersaturated), cooled to $T_f = 273.15$ K in 120 min and held for 30 min.

**Decision variables**:
* seed loading, `seed_load` (% of the dissolved C). The seeds are log-normally distributed around 100 $\mu$m.
* shape of the cooling profile, `p`:

$$ T(t) = T_f + (T_0 - T_f) \left[1 - \left(\frac{t}{t_{cool}}\right)^p\right] $$

`p = 1` is linear cooling, `p > 1` cools slowly at the beginning and faster at the end (the "cubic" profile often recommended for seeded batches), and `p < 1` does the opposite.

**Seeds on the FVM grid.** The seed distribution is given as a volume-fraction shape plus a seed mass (`distrib_basis='vol_perc'`), and PharmaPy converts it into a number density.

**Building a crystallizer** follows the same recipe as the reactor: the unit (with its integrator and, here, a temperature controller), then the phases, then the kinetics. The solid phase carries the population balance as a *mechanism*, which must be attached **before** the solid goes into the crystallizer.

In [ ]:
T0, TF = 293.15, 273.15               # K
T_COOL, T_HOLD = 120 * 60, 30 * 60     # s
SEED_SIZE = 100                        # um


def cooling_profile(p):
    def temp_func(t):
        frac = min(t / T_COOL, 1.0)
        return TF + (T0 - TF) * (1 - frac**p)
    return temp_func


def run_batch(seed_load, p, nucl_sec=NO_SECONDARY, mass=1.0):
    """Seeded batch cooling crystallizer. seed_load in % of the dissolved C."""
    controller = ...  # <-- COMPLETE THIS: SimpleTemperatureController following cooling_profile(p)
    cryst = ...  # <-- COMPLETE THIS: BatchCrystallizer with its own ScipyBackend (maxh 60 s) and the controller

    liquid = ...  # <-- COMPLETE THIS: 'mass' kg of liquid at T0 with mass_frac=[0, 0, W_C, 0, 1 - W_C] (name_solv='solvent')
    solid = ...  # <-- COMPLETE THIS: empty solid phase of pure C at T0 (mass=0, mass_frac=[0, 0, 1, 0, 0])

    seed_mass = seed_load / 100 * W_C * mass                               # kg
    seed_shape = np.exp(-0.5 * (np.log(X_GRID / SEED_SIZE) / 0.15)**2)     # volume-fraction shape on X_GRID

    # <-- COMPLETE THIS: attach a OneDFVMMechanism on X_GRID to 'solid': the seed shape if seed_mass > 0 (distrib_basis='vol_perc', basis_mass=seed_mass, vol_slurry=liquid.vol), else an empty distribution
    ...

    cryst.Phases = ...  # <-- COMPLETE THIS: load the liquid and the solid into the crystallizer
    cryst.CrystKinetics = ...  # <-- COMPLETE THIS: attach a new kinetics object (after the phases)

    ...  # <-- COMPLETE THIS: solve on 61 points from 0 to T_COOL + T_HOLD (verbose=False)
    return cryst


def csd_kpis(cryst):
    mech = ...  # <-- COMPLETE THIS: the population balance (mechanism) of the first solid phase
    distrib = ...  # <-- COMPLETE THIS: number density at the final time (result field distrib_solid0)
    kpis = ...  # <-- COMPLETE THIS: d10, d50, d90 and span of the final distribution (csd_quantiles)
    number = distrib * mech.dx
    kpis['mean_size'] = np.sum(number * mech.x_grid) / np.sum(number)   # mu_1 / mu_0
    return kpis

### Base case: unseeded, linear cooling

In [ ]:
tic = time.perf_counter()
base = run_batch(seed_load=0, p=1)
print('simulation time: %.1f s' % (time.perf_counter() - tic))
print({key: round(val, 2) for key, val in csd_kpis(base).items()})


def plot_batch(cryst, label, axes):
    res = cryst.result
    axes[0].plot(res.time / 60, res.global_temp, label=label)
    axes[1].plot(res.time / 60, np.ravel(res.supersat), label=label)
    axes[2].semilogx(*volume_density(cryst), label=label)

    axes[0].set_ylabel('T (K)')
    axes[1].set_ylabel('supersaturation S (-)')
    axes[2].set_ylabel(r'volume density, $q_3(\ln L)$')
    axes[0].set_xlabel('time (min)')
    axes[1].set_xlabel('time (min)')
    axes[2].set_xlabel(r'size ($\mu$m)')


fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
plot_batch(base, 'unseeded, p = 1', axes)
axes[2].legend()
fig.tight_layout()
plt.show()

### Effect of seeding
The full CSD shows *how* seeding changes the shape of the distribution, not only its average.

In [ ]:
seed_loads = [0, 0.5, 1, 2, 5, 10]
scan, runs = [], {}
for load in seed_loads:
    runs[load] = run_batch(seed_load=load, p=1)
    scan.append({'seed_load (%)': load, **csd_kpis(runs[load])})

scan = pd.DataFrame(scan).set_index('seed_load (%)')
print(scan.round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(scan.index, scan['d50'], 'o-', label='$d_{50}$')
axes[0].plot(scan.index, scan['mean_size'], 's--', label=r'$\mu_1/\mu_0$')
axes[1].plot(scan.index, scan['span'], 'o-')
for load in [0, 1, 5]:
    axes[2].semilogx(*volume_density(runs[load]), label='%g%% seed' % load)
axes[0].set_ylabel(r'size ($\mu$m)')
axes[0].legend()
axes[1].set_ylabel('span (-)')
axes[0].set_xlabel('seed loading (%)')
axes[1].set_xlabel('seed loading (%)')
axes[2].set_xlabel(r'size ($\mu$m)')
axes[2].set_ylabel(r'$q_3(\ln L)$')
axes[2].legend()
fig.tight_layout()
plt.show()

Little seed leaves room for nucleation, which gives a **bimodal** CSD (large grown seeds + fines) and a large span. A lot of seed suppresses nucleation (narrow CSD), but the solute is shared among more crystals, so each grows less. Note also how different the number-average size $\mu_1/\mu_0$ is from $d_{50}$: fines dominate the number of crystals, but not their volume.

### Optimization

We combine both goals in a weighted-sum objective (to be **minimized**):

$$ J = - w_{size} \frac{d_{50}}{100\ \mu m} + w_{span}\, \mathrm{span} $$

and solve it with the derivative-free Nelder-Mead method. The bounds are enforced by clipping the decision variables plus a quadratic penalty. **Try changing the weights** and see how the optimal recipe changes.

In [ ]:
W_SIZE, W_SPAN = 1.0, 1.0

BOUNDS_BATCH = np.array([[0.2, 10.0],    # seed_load (%)
                         [0.3, 3.0]])    # cooling exponent p

history_batch = []


def objective_batch(x):
    x_clip = np.clip(x, BOUNDS_BATCH[:, 0], BOUNDS_BATCH[:, 1])
    penalty = 10 * np.sum(((x - x_clip) / (BOUNDS_BATCH[:, 1] - BOUNDS_BATCH[:, 0]))**2)

    kpis = ...  # <-- COMPLETE THIS: KPIs of the batch run at x_clip (seed_load, p)
    J = ...  # <-- COMPLETE THIS: weighted-sum objective (maximize d50, minimize span)

    history_batch.append({'seed_load': x_clip[0], 'p': x_clip[1], **kpis, 'J': J})
    return J + penalty


tic = time.perf_counter()
# <-- COMPLETE THIS: call scipy's minimize on objective_batch with Nelder-Mead from x0 = [2, 1] (hint: options={'maxfev': 20, 'initial_simplex': [[2, 1], [5, 1], [2, 2.5]]})
...
print('optimization time: %.0f s' % (time.perf_counter() - tic))

history_batch = pd.DataFrame(history_batch)
best_batch = history_batch.loc[history_batch['J'].idxmin()]
print(best_batch.round(3))

In [ ]:
optimal = run_batch(best_batch['seed_load'], best_batch['p'])

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
plot_batch(base, 'unseeded, linear', axes)
plot_batch(optimal, 'optimum: %.1f%% seed, p = %.2f' % (best_batch['seed_load'], best_batch['p']), axes)
axes[2].legend()
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 3.5))
sc = ax.scatter(history_batch['seed_load'], history_batch['p'], c=history_batch['J'], cmap='viridis')
ax.plot(history_batch['seed_load'], history_batch['p'], 'k-', alpha=0.2)
fig.colorbar(sc, label='J')
ax.set_xlabel('seed loading (%)')
ax.set_ylabel('cooling exponent p')
ax.set_title('Nelder-Mead iterates')
plt.show()

Numerical diffusion of the first-order FVM widens the CSD somewhat: re-run the optimum with 100 or 200 grid cells and compare the span.

---
# Part B: How do MSMPRs compare?

**Operation**: a 20 wt% C solution at 313 K is fed continuously; the last MSMPR is at 273.15 K (same yield as the batch), and the total residence time is 90 min. We compare a few designs:
* one MSMPR at 273.15 K
* two MSMPRs, $T_1$ = 283.15 K and equal volumes
* two MSMPRs, $T_1$ = 288.15 K and equal volumes

**Building a cascade.** Units are connected with `SimulationExec` and a flowsheet string (`'CR01 --> CR02'`). Only the first MSMPR gets an `Inlet`; PharmaPy passes the outlet slurry of `CR01` to `CR02`. Three practical details:
* Each MSMPR is started up full of solution **saturated** at its temperature, which avoids a huge nucleation burst at $t = 0$.
* `stall_calls=20000` makes the scipy integrator give up quickly (instead of after minutes) if it ever gets stuck.
* We simulate 12 total residence times so that the whole distribution reaches steady state. A cascade simulation takes 10-30 s, so instead of an optimization loop we compare the designs above.

In [ ]:
TAU_TOTAL = 90 * 60     # s
V_TOTAL = 2e-3          # m3
T_FEED = 313.15         # K
N_TAU = 12


def saturated_liquid(T, vol, nucl_sec=NO_SECONDARY):
    c_sat = ...  # <-- COMPLETE THIS: solubility of C at T (kg C / m3 solvent)
    w_sat = min(c_sat / (c_sat + RHO_SOLVENT), W_C)
    return ...  # <-- COMPLETE THIS: liquid of volume 'vol' at T with mass_frac=[0, 0, w_sat, 0, 1 - w_sat]


def make_msmpr(T, vol, nucl_sec=NO_SECONDARY):
    # <-- COMPLETE THIS: a ContinuousCrystallizer at constant T (ScipyBackend(options={'maxh': 600}, stall_calls=20000) and ContinuousVesselController(temp_func=...)), and an empty solid phase of C at T
    ...
    # <-- COMPLETE THIS: attach an empty OneDFVMMechanism (same X_GRID) to 'solid', then load the phases (saturated liquid first) and the kinetics
    ...
    return cryst


def run_cascade(temps, fracs, nucl_sec=NO_SECONDARY):
    """MSMPRs in series at temperatures 'temps' with volume fractions 'fracs'. Returns the flowsheet."""
    names = ['CR%02d' % (ind + 1) for ind in range(len(temps))]
    flst = ...  # <-- COMPLETE THIS: SimulationExec connecting the units in 'names' in series
    for name, T, frac in zip(names, temps, fracs):
        ...  # <-- COMPLETE THIS: add an MSMPR at T with volume frac * V_TOTAL as unit 'name'

    # <-- COMPLETE THIS: feed the first crystallizer: a LiquidStream at T_FEED with vol_flow=V_TOTAL / TAU_TOTAL and mass_frac=[0, 0, W_C, 0, 1 - W_C]
    ...

    time_grid = np.linspace(0, N_TAU * TAU_TOTAL, 25)
    # <-- COMPLETE THIS: solve the flowsheet with the same time_grid for every unit (kwargs_run={name: {'time_grid': time_grid, 'verbose': False}, ...}, verbose=False)
    ...
    return flst


def product(flst):
    return ...  # <-- COMPLETE THIS: the last crystallizer of the flowsheet (flst.execution_names lists the units in order)


designs = {'1 MSMPR': ([273.15], [1.0]),
           '2 MSMPRs, T1 = 283 K': ([283.15, 273.15], [0.5, 0.5]),
           '2 MSMPRs, T1 = 288 K': ([288.15, 273.15], [0.5, 0.5])}

tic = time.perf_counter()
cascades = {label: run_cascade(*design) for label, design in designs.items()}
print('simulation time: %.0f s' % (time.perf_counter() - tic))

comparison = pd.DataFrame({'batch optimum': csd_kpis(optimal),
                           **{label: csd_kpis(product(flst)) for label, flst in cascades.items()}})
print(comparison.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(*volume_density(optimal), 'k', lw=2, label='batch optimum')
for label, flst in cascades.items():
    ax.semilogx(*volume_density(product(flst)), label=label)
ax.set_xlabel(r'size ($\mu$m)')
ax.set_ylabel(r'volume density, $q_3(\ln L)$')
ax.legend()
plt.show()

With size-independent growth and nuclei born at $L_0$, the steady-state CSD of an ideal MSMPR is **always exponential**, $n(L) = n_0\, e^{-L/G\tau}$, whatever the kinetics. Its volume-weighted span is therefore fixed at about 1.34. A cascade narrows it only slightly. In this system, where only **primary** nucleation is active, the seeded batch nucleates once and then grows the crystals, so it gives a much narrower CSD than any MSMPR.

---
# Part C: Secondary nucleation

In industrial crystallizers, **secondary nucleation** (crystal-crystal and crystal-impeller contacts) usually dominates over primary nucleation. It scales with the amount of crystals present, $B_s = k_s S^{s_1} (k_v \mu_3)^{s_2}$, and keeps producing nuclei at low supersaturation. Let's switch it on and repeat the batch optimum and the single MSMPR (about 30 s).

In [ ]:
SECONDARY = (5e12, 0, 2, 1)   # (k_s, E_s, s1, s2): Bs = k_s * S^2 * (kv * mu3)

tic = time.perf_counter()
batch_sec = ...  # <-- COMPLETE THIS: the batch optimum from Part A, with secondary nucleation
msmpr_sec = ...  # <-- COMPLETE THIS: the single MSMPR at 273.15 K, with secondary nucleation
print('simulation time: %.0f s' % (time.perf_counter() - tic))

secondary = pd.DataFrame({
    'batch optimum, primary only': csd_kpis(optimal),
    'batch optimum, + secondary': csd_kpis(batch_sec),
    '1 MSMPR, primary only': csd_kpis(product(cascades['1 MSMPR'])),
    '1 MSMPR, + secondary': csd_kpis(msmpr_sec),
})
print(secondary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].semilogx(*volume_density(optimal), label='primary only')
axes[0].semilogx(*volume_density(batch_sec), label='+ secondary')
axes[0].set_title('seeded batch (Part A optimum)')
axes[1].semilogx(*volume_density(product(cascades['1 MSMPR'])), label='primary only')
axes[1].semilogx(*volume_density(msmpr_sec), label='+ secondary')
axes[1].set_title('1 MSMPR')
for ax in axes:
    ax.set_xlabel(r'size ($\mu$m)')
    ax.legend()
axes[0].set_ylabel(r'volume density, $q_3(\ln L)$')
plt.show()

**Discussion**
* **Secondary nucleation broadens the batch CSD dramatically.** Once crystals are present they keep generating new nuclei throughout the batch, so the batch loses the "one burst, then grow" behaviour that made it narrow. The recipe optimized without secondary nucleation is no longer optimal.
* **The MSMPR span does not change** (still about 1.3): only $G\tau$, and therefore the size, changes. With dominant secondary nucleation the continuous process becomes the one with the *narrower* CSD, which is what is often observed in practice.
* Which process looks better is therefore a property of the **kinetics**, not only of the equipment. That is a good reason to estimate nucleation kinetics carefully before choosing between batch and continuous.

**Things to try**
* Re-optimize the batch (Part A) with `nucl_sec=SECONDARY`. Can seeding or cooling still recover a narrow CSD?
* Vary $k_s$ between `5e10` and `5e13`. Where does the batch become broader than the MSMPR?
* Change the weights `W_SIZE`, `W_SPAN` and compare the batch optimum with the MSMPR designs again.
* Make the seed size a third decision variable. Why is the answer then "as large as allowed"?
* Change `T_COOL`. Is a longer batch always better?